## SNOTEL QC Evaluation
Evaluation of PNNL Snotel data quality and consistency.

In [1]:
%matplotlib ipympl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, date
from buildforcing.datasets import PNNLSnotel

### SNOTEL missing data
I want to get a sense for how much data is typically missing in the SNOTEL data.

In [3]:
snotel = PNNLSnotel(site_name='Camp Jackson', storage_path='C:/Users/clmbn/NMT_PhD/data/snotel/')
snotel.load_data()

In [4]:
start_date, end_date = snotel.find_usable_dates()
print(f"Usable dates: {start_date} to {end_date}")

Usable dates: 2002-10-01 to 2021-09-30


In [5]:
# Extract SNOTEL data for the usable date range or starting in 1990, whichever is later
if start_date < date(1990, 1, 1):
    start_date = date(1990, 1, 1)

snotel_data = snotel.data[start_date.strftime('%Y-%m-%d'):end_date.strftime('%Y-%m-%d')]

In [ ]:
# Ensure index is daily
snotel_data = snotel_data.asfreq('D')

In [ ]:
# Calculate statistics for the number of sequential missing days in T max
def count_sequential_missing_days(data: np.array) -> tuple:
    '''
    Count the number of sequential missing days in numpy array data. Assumes data is daily over input.
    Returns two lists: streak_lengths and streak_counts.
    '''
    missing_stats = {}
    current_streak = 0
    for day in data:
        if pd.isna(day):
            current_streak += 1
        else:
            if current_streak > 0:
                if current_streak in missing_stats:
                    missing_stats[current_streak] += 1
                else:
                    missing_stats[current_streak] = 1
            current_streak = 0
    # Check if there was a streak at the end of the data
    if current_streak > 0:
        if current_streak in missing_stats:
            missing_stats[current_streak] += 1
        else:
            missing_stats[current_streak] = 1

    # Sort the output dictionary by keys (number of missing days)
    missing_stats = dict(sorted(missing_stats.items()))
    # Convert keys and values to lists
    streak_lengths = list(missing_stats.keys())
    streak_counts = list(missing_stats.values())
    return streak_lengths, streak_counts

In [ ]:
streaks_max, streak_counts_max = count_sequential_missing_days(snotel_data['T_max_C'].values)
streaks_min, streak_counts_min = count_sequential_missing_days(snotel_data['T_min_C'].values)

#Print the results
print("Max T streaks:", streaks_max)
print("Max T counts:", streak_counts_max)
print("Min T streaks:", streaks_min)
print("Min T counts:", streak_counts_min)

plt.figure(figsize=(12, 6))
plt.bar(streaks_max, streak_counts_max, width=1.0, alpha=0.7, label='T_max_C')
plt.bar(streaks_min, streak_counts_min, width=0.5, alpha=0.7, label='T_min_C')
plt.xlabel('Missing streak length (days)')
plt.ylabel('Count')
plt.legend()
plt.title('Sequential Missing Days in SNOTEL Data')
plt.show()




In [ ]:
# Missing data in precipitation and swe
streaks_swe, streak_counts_swe = count_sequential_missing_days(snotel_data['swe_mm'].values)
streaks_precip, streak_counts_precip = count_sequential_missing_days(snotel_data['precip_mm'].values)

#Print the results
print("Max SWE streaks:", streaks_swe)
print("Max SWE counts:", streak_counts_swe)
print("Max Precip streaks:", streaks_precip)
print("Max Precip counts:", streak_counts_precip)

plt.figure(figsize=(12, 6))
plt.bar(streaks_swe, streak_counts_swe, width=1.0, alpha=0.7, label='SWE_mm')
plt.bar(streaks_precip, streak_counts_precip, width=0.5, alpha=0.7, label='Precip_mm')
plt.xlabel('Missing streak length (days)')
plt.ylabel('Count')
plt.legend()
plt.title('Sequential Missing Days in SNOTEL Data')
plt.show()

In [ ]:
# Calculate F temperatures for easier analysis
snotel_data['T_max_F'] = snotel_data['T_max_C'] * 9/5 + 32
snotel_data['T_min_F'] = snotel_data['T_min_C'] * 9/5 + 32

In [ ]:
# Create a single figure with 3 subplots (temperature, SWE, precipitation)
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Plot temperatures in the top subplot
snotel_data['T_max_F'].plot(ax=ax1, label='T_max_C', color='red', alpha=0.7)
snotel_data['T_min_F'].plot(ax=ax1, label='T_min_C', color='blue', alpha=0.7)
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('SNOTEL Temperature Data')
ax1.legend()
ax1.grid(True)

# Plot SWE in the middle subplot
ax2.plot(snotel_data.index, snotel_data['swe_mm'], color='blue')
ax2.set_ylabel('SWE (mm)')
ax2.set_title('Snow Water Equivalent')
ax2.grid(True)

# Plot Precipitation in the bottom subplot
ax3.plot(snotel_data.index, snotel_data['precip_mm'], color='green')
ax3.set_xlabel('Date')
ax3.set_ylabel('Precipitation (mm)')
ax3.set_title('Precipitation')
ax3.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# print latitude and longitude
print(f"Latitude: {snotel.latitude}, Longitude: {snotel.longitude}")